In [60]:
# Importing Libraries
import pandas as pd
from src.utils.db_connector import get_sqlalchemy_engine
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import joblib
import numpy as np
from sklearn.utils import compute_class_weight
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasClassifier


In [3]:
# Data Importing
ML_TABLE = "gold.T_ML_Customer_Features"
try:
    Engine=get_sqlalchemy_engine()
    sql_query=f"SELECT * FROM {ML_TABLE}"
    df_ml = pd.read_sql(sql_query, Engine)
except Exception as e:
    print("Error Raised : ", e)

In [33]:
# Data Spliting
numerical_features=[
    'recency_days',
    'total_revenue_spent',
    'max_payment_installments',
    'avg_item_weight_g'
]
categorical_features=[
    'customer_state',
    'most_freq_category'
]
columns_to_drop = [
    'customer_unique_id',
    'avg_customer_review',
    'Y_satisfaction_class',
    'customer_city',
    'customer_lat',
    'customer_lng'
]
Y = df_ml['Y_satisfaction_class']
X = df_ml.drop(columns=columns_to_drop,errors='ignore')
print(f"Target Y successfully defined as multi-class (0, 1, 2). Target size: {Y.shape}")
print(f"Target X successfully defined Independent Var size: {X.shape}")

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)
print(f"Training set size: {X_train.shape[0]:,} samples")
print(f"Testing set size: {X_test.shape[0]:,} samples")


Target Y successfully defined as multi-class (0, 1, 2). Target size: (94029,)
Target X successfully defined Independent Var size: (94029, 8)
Training set size: 75,223 samples
Testing set size: 18,806 samples


In [59]:
classes = np.unique(Y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=Y_train)
class_weights = dict(zip(classes, weights))
Y_train_weights = np.array([class_weights[y] for y in Y_train])

In [61]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)
NUM_FEATURES = X_train.shape[1]
NUM_CLASSES = 3

def create_mlp_model(input_dim=NUM_FEATURES, num_classes=NUM_CLASSES):
    model = Sequential()

    model.add(Dense(64, input_dim=input_dim, activation='relu'))

    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])
    return model

keras_classifier = KerasClassifier(
    model=create_mlp_model,
    epochs=50,
    batch_size=32,
    verbose=0
)

nn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', keras_classifier)
])

nn_pipeline.fit(
    X_train,
    Y_train,
    classifier__class_weight=class_weights
)

C:\Users\Ayush\anaconda3\envs\Purva_Patole\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense" is incompatible with the layer: expected axis -1 of input shape to have value 8, but received input with shape (None, 104)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 104), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [53]:
# Testing Phase
def TestPhase(X_test, Y_test,Y_pred,rf_pipeline):
    Y_pred = rf_pipeline.predict(X_test)

    print("\n--- Model Evaluation (Test Set) ---")

    print(classification_report(Y_test, Y_pred, target_names=['0:Bad', '1:Satisfied', '2:Good']))

    accuracy = accuracy_score(Y_test, Y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    # 4. Confusion Matrix
    cm = confusion_matrix(Y_test, Y_pred)
    cm_df = pd.DataFrame(cm,
                         index=['Actual Bad', 'Actual Satisfied', 'Actual Good'],
                         columns=['Predicted Bad', 'Predicted Satisfied', 'Predicted Good'])
    print("\nConfusion Matrix (Rows=Actual, Columns=Predicted):")
    display(cm_df)
TestPhase(X_test,Y_test,Y_pred,nn_pipeline)

C:\Users\Ayush\anaconda3\envs\Purva_Patole\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Model Evaluation (Test Set) ---
              precision    recall  f1-score   support

       0:Bad       0.31      0.40      0.35      4190
 1:Satisfied       0.21      0.31      0.25      3632
      2:Good       0.63      0.47      0.54     10984

    accuracy                           0.42     18806
   macro avg       0.39      0.39      0.38     18806
weighted avg       0.48      0.42      0.44     18806


Overall Accuracy: 0.4219

Confusion Matrix (Rows=Actual, Columns=Predicted):


,Predicted Bad,Predicted Satisfied,Predicted Good
Actual Bad,1685,1074,1431
Actual Satisfied,982,1108,1542
Actual Good,2688,3154,5142


In [55]:
# HyperParameter
tuned_classifier = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    n_estimators=500,              # Increased trees
    learning_rate=0.01,            # Reduced learning rate (was 0.05)
    max_depth=7,                   # Maintained depth (5 is safe for generalization)
    gamma=1,                     # NEW: Added Gamma regularization (Min loss reduction needed to make a split)
    reg_lambda=2,                # NEW: Added L2 regularization (Controls complexity)
    eval_metric='mlogloss',
    random_state=42
)
tuned_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', tuned_classifier)
])



print("\n--- Starting Final Hyperparameter Tuning and Retraining ---")
tuned_pipeline.fit(
    X_train,
    Y_train,
    classifier__sample_weight=Y_train_weights
)
print("✅ Hyperparameter Tuning Retraining Complete.")




--- Starting Final Hyperparameter Tuning and Retraining ---
✅ Hyperparameter Tuning Retraining Complete.


In [56]:
TestPhase(X_test,Y_test,Y_pred)

C:\Users\Ayush\anaconda3\envs\Purva_Patole\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Model Evaluation (Test Set) ---
              precision    recall  f1-score   support

       0:Bad       0.31      0.40      0.35      4190
 1:Satisfied       0.21      0.31      0.25      3632
      2:Good       0.63      0.47      0.54     10984

    accuracy                           0.42     18806
   macro avg       0.39      0.39      0.38     18806
weighted avg       0.48      0.42      0.44     18806


Overall Accuracy: 0.4219

Confusion Matrix (Rows=Actual, Columns=Predicted):


,Predicted Bad,Predicted Satisfied,Predicted Good
Actual Bad,1685,1074,1431
Actual Satisfied,982,1108,1542
Actual Good,2688,3154,5142
